In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
df_inventory_raw = spark.read.parquet("abfss://e8b72d8d-5a0f-40b4-bcfc-e5ff1552d786@onelake.dfs.fabric.microsoft.com/ac40a136-0e9a-48e0-ab5d-652add7f1fed/Files/bronze/inventory_data")
display(df_inventory_raw)

1. Rename the Required Columns

In [ ]:
df_inventory=df_inventory_raw.withColumnRenamed("productName", "ProductName").withColumnRenamed("cost_price", "CostPrice").withColumnRenamed("last_stocked", "LastStockedDate")

display(df_inventory)


2. Change available from [Yes, No], [y,n], [true, false] into [Yes , No] or [1,0]

In [ ]:
from pyspark.sql import functions as F

df_inventory = df_inventory.withColumn(
    "available",
    F.when(F.col("available").isNull(), None)
    .when(F.lower(F.col("available")).isin("yes", "y", "true", "1"), "Yes")
    .otherwise("No")
)
display(df_inventory)

In [ ]:
%%html
2. Required Changes in Cost Price

In [ ]:
from pyspark.sql import functions as F

# Pattern matches $, usd, rs, inr (case-insensitive via (?i))
currency_pattern = r"(?i)\$|₹|usd|rs|inr"
 
# CostPrice
# st1. replace currency Pattern in Col
df_inventory = df_inventory.withColumn(
    "CostPrice", 
    F.trim(F.regexp_replace(F.col("CostPrice"), currency_pattern, ""))
)

#st2. Replace "." with empty value
df_inventory = df_inventory.withColumn(
    "CostPrice", 
    F.regexp_replace(F.col("CostPrice"), r"\.", "")
)

# st3. change type to int
#Using IntegerType (requires import)
from pyspark.sql.types import IntegerType
df_inventory = df_inventory.withColumn("CostPrice", df_inventory["CostPrice"].cast(IntegerType()))
display(df_inventory)

In [ ]:
display(df_inventory)

3.  Change Date format to standard date format

In [ ]:
from pyspark.sql.functions import col, coalesce, to_date

# Overwrite the existing column with the standardized date type
df_inventory = df_inventory.withColumn(
    "LastStockedDate",
    coalesce(
        to_date(col("LastStockedDate"), "yyyy-MM-dd"),
        to_date(col("LastStockedDate"), "yyyy/MM/dd"),
        to_date(col("LastStockedDate"), "yyyy.MM.dd"),
        to_date(col("LastStockedDate"), "dd-MM-yyyy"),
        to_date(col("LastStockedDate"), "dd/MM/yyyy"),
        to_date(col("LastStockedDate"), "dd.MM.yyyy"),
        to_date(col("LastStockedDate"), "MM/dd/yyyy")
    )
)

# Verify the changes and the updated schema
df_inventory.show(15)
df_inventory.printSchema()


In [ ]:
df_stock=df_inventory.select('stock').distinct()
display(df_stock)

In [ ]:
from pyspark.sql.functions import col, when, regexp_extract, lower, trim
from pyspark.sql.types import IntegerType

# Clean whitespace and force lowercase for consistent text mapping
cleaned_stock = lower(trim(col("Stock")))

df_inventory = df_inventory.withColumn(
    "Stock",
    when(cleaned_stock == "fifteen", 15)
    .when(cleaned_stock == "eighteen", 18)
    .when(cleaned_stock == "twenty", 20)
    .when(cleaned_stock == "twenty five", 25)
    # Extract the first sequence of digits (\d+) for mixed entries like "25 units"
    .otherwise(regexp_extract(col("Stock"), r"(\d+)", 1).cast(IntegerType()))
)

# Verify the updated schema and values
df_inventory.show(15)
df_inventory.printSchema()


In [ ]:
from pyspark.sql.functions import col, initcap, regexp_replace
df_inventory=df_inventory.withColumn("Warehouse", 
        initcap(trim(regexp_replace(col("warehouse"), r"[^a-zA-Z0-9\s]", " ")))
    )
display(df_inventory)

In [ ]:
# Optional: Save to Silver Layer
df_inventory.write.mode("overwrite").format("delta").saveAsTable("silver_inventory")